In [2]:
%pip install folium geopandas branca


   ------------- -------------------------- 1/3 [branca]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
print(sys.executable)

C:\Users\aafir\AppData\Local\Programs\Python\Python313\python.exe


In [1]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import glob
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins

DATA = "data/"

# ---------- Load ----------
study_area   = gpd.read_file(DATA + "study_area.geojson")
settlements  = gpd.read_file(DATA + "settlements.geojson")
rivers       = gpd.read_file(DATA + "rivers.geojson")
normal_water = gpd.read_file(glob.glob(DATA + "normal_water_*.geojson")[0])
flood_extent = gpd.read_file(glob.glob(DATA + "flood_extent_*.geojson")[0])

river_gauges  = pd.read_csv(DATA + "river_gauges.csv")
gauge_levels  = pd.read_csv(DATA + "river_gauge_levels_july2026.csv")
relief_camps  = pd.read_csv(DATA + "relief_camps.csv")
rain_stations = pd.read_csv(DATA + "rainfall_stations.csv")
rain_daily    = pd.read_csv(DATA + "rainfall_daily_july2026.csv")

# ---------- Make all vector layers lat/lon ----------
def to_wgs84(g):
    return g.to_crs(4326) if g.crs else g.set_crs(4326)

study_area, settlements, rivers, normal_water, flood_extent = [
    to_wgs84(g) for g in (study_area, settlements, rivers, normal_water, flood_extent)
]

# ---------- Helpers ----------
def find_col(df, options):
    for c in df.columns:
        if c.lower() in options:
            return c
    raise KeyError(f"None of {options} found. Columns: {list(df.columns)}")

def points_layer(m, df, name, color, icon):
    lat = find_col(df, {"lat", "latitude", "y"})
    lon = find_col(df, {"lon", "lng", "long", "longitude", "x"})
    fg = folium.FeatureGroup(name=name)
    for _, r in df.dropna(subset=[lat, lon]).iterrows():
        tip = "<br>".join(f"<b>{c}</b>: {r[c]}" for c in df.columns[:6])
        folium.Marker([r[lat], r[lon]], tooltip=folium.Tooltip(tip),
                      icon=folium.Icon(color=color, icon=icon)).add_to(fg)
    fg.add_to(m)

# ---------- Map ----------
bounds = study_area.total_bounds  # minx, miny, maxx, maxy
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

m = folium.Map(location=center, zoom_start=10, tiles="CartoDB positron")

folium.GeoJson(study_area, name="Study area",
    style_function=lambda f: {"color": "black", "weight": 2, "fillOpacity": 0}).add_to(m)

folium.GeoJson(normal_water, name="Normal water (20 May 2026)",
    style_function=lambda f: {"color": "#1f78b4", "fillColor": "#a6cee3",
                              "fillOpacity": 0.6, "weight": 1}).add_to(m)

folium.GeoJson(flood_extent, name="Flood extent (10 Jul 2026)",
    style_function=lambda f: {"color": "#e31a1c", "fillColor": "#fb9a99",
                              "fillOpacity": 0.5, "weight": 1}).add_to(m)

folium.GeoJson(rivers, name="Rivers",
    style_function=lambda f: {"color": "#0b5394", "weight": 2}).add_to(m)

settle_fg = folium.FeatureGroup(name="Settlements")
for _, r in settlements.iterrows():
    pt = r.geometry.representative_point()
    tip = "<br>".join(f"<b>{c}</b>: {r[c]}" for c in settlements.columns if c != "geometry")
    folium.CircleMarker([pt.y, pt.x], radius=4, color="#333", fill=True,
                        fill_opacity=0.8, tooltip=folium.Tooltip(tip)).add_to(settle_fg)
settle_fg.add_to(m)

points_layer(m, relief_camps,  "Relief camps",      "green",  "home")
points_layer(m, river_gauges,  "River gauges",      "blue",   "tint")
points_layer(m, rain_stations, "Rainfall stations", "purple", "cloud")

m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
folium.LayerControl(collapsed=False).add_to(m)
plugins.Fullscreen().add_to(m)
plugins.MiniMap().add_to(m)

m.save("flood_map.html")
m

C:\Users\aafir\AppData\Local\Programs\Python\Python313\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [3]:
aoi        = gpd.read_file(DATA + "study_area.geojson")
rivers     = gpd.read_file(DATA + "rivers.geojson")
flood      = gpd.read_file(DATA + "flood_extent_2026-07-10.geojson")
normal     = gpd.read_file(DATA + "normal_water_2026-05-20.geojson")
settle     = gpd.read_file(DATA + "settlements.geojson")

rain_st    = pd.read_csv(DATA + "rainfall_stations.csv")
rain_daily = pd.read_csv(DATA + "rainfall_daily_july2026.csv")
gauges     = pd.read_csv(DATA + "river_gauges.csv")
gauge_ts   = pd.read_csv(DATA + "river_gauge_levels_july2026.csv")
camps      = pd.read_csv(DATA + "relief_camps.csv")

rain_gdf  = gpd.GeoDataFrame(rain_st, geometry=gpd.points_from_xy(rain_st.lon, rain_st.lat), crs="EPSG:4326")
gauge_gdf = gpd.GeoDataFrame(gauges,  geometry=gpd.points_from_xy(gauges.lon, gauges.lat),  crs="EPSG:4326")

for name, g in [("rivers", rivers), ("flood", flood), ("settlements", settle), ("rain", rain_gdf), ("gauges", gauge_gdf)]:
    print(f"{name:12s} {len(g):3d} features | CRS {g.crs}")

rivers         6 features | CRS EPSG:4326
flood          3 features | CRS EPSG:4326
settlements   20 features | CRS EPSG:4326
rain          12 features | CRS EPSG:4326
gauges         7 features | CRS EPSG:4326


In [4]:
affected = gpd.sjoin(settle, flood[["depth_class", "severity", "geometry"]], how="left", predicate="within")
affected["flooded"] = affected["depth_class"].notna()
affected["depth_class"] = affected["depth_class"].fillna("Not flooded")

flood_utm = flood.to_crs("EPSG:32646")
flood["area_km2"] = (flood_utm.area / 1e6).round(1)
normal_km2 = normal.to_crs("EPSG:32646").area.sum() / 1e6
total_flood_km2 = flood["area_km2"].sum()

print(f"Total flooded area : {total_flood_km2:,.0f} km²")
print(f"Normal water area  : {normal_km2:,.0f} km²")
print(f"Settlements flooded: {affected.flooded.sum()} of {len(affected)}")
print(f"Population exposed : {affected.loc[affected.flooded, 'population'].sum():,}")
affected.loc[affected.flooded, ["name", "district", "population", "depth_class"]]

Total flooded area : 4,532 km²
Normal water area  : 710 km²
Settlements flooded: 7 of 20
Population exposed : 204,000


,name,district,population,depth_class
1,North Guwahati,Kamrup,32800,Shallow (<0.5 m)
2,Tezpur,Sonitpur,103200,Deep (>1.5 m)
4,Morigaon,Morigaon,24100,Shallow (<0.5 m)
9,Dhing,Nagaon,21300,Deep (>1.5 m)
13,Bhuragaon,Morigaon,8100,Shallow (<0.5 m)
14,Laharighat,Morigaon,6800,Deep (>1.5 m)
19,Silghat,Nagaon,7700,Deep (>1.5 m)


In [5]:
minx, miny, maxx, maxy = aoi.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=9, tiles=None, control_scale=True)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri Light Gray", name="Light grey (Esri)").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite (Esri)").add_to(m)

folium.GeoJson(aoi, name="Study area",
               style_function=lambda f: {"color": "#0F172A", "weight": 2, "dashArray": "6 4", "fillOpacity": 0}).add_to(m)
m

In [6]:
flood_colors = {1: "#93C5FD", 2: "#3B82F6", 3: "#1E3A8A"}

folium.GeoJson(normal, name="Normal water (20 May 2026)",
               style_function=lambda f: {"fillColor": "#06B6D4", "color": "#0891B2", "weight": 0.5, "fillOpacity": 0.6}
               ).add_to(m)

folium.GeoJson(flood, name="Flood extent (10 Jul 2026)",
               style_function=lambda f: {"fillColor": flood_colors[f["properties"]["severity"]],
                                         "color": flood_colors[f["properties"]["severity"]],
                                         "weight": 0.5, "fillOpacity": 0.55},
               highlight_function=lambda f: {"weight": 2, "color": "#F59E0B"},
               tooltip=folium.GeoJsonTooltip(fields=["depth_class", "area_km2", "date"],
                                             aliases=["Depth class", "Area (km²)", "Date"])
               ).add_to(m)
m

In [7]:
width = {1: 5, 2: 3, 3: 2}
folium.GeoJson(rivers, name="River network",
               style_function=lambda f: {"color": "#0369A1", "weight": width[f["properties"]["order"]]},
               tooltip=folium.GeoJsonTooltip(fields=["river_name", "type", "length_km"],
                                             aliases=["River", "Type", "Length in AOI (km)"])
               ).add_to(m)

In [8]:
imd_colors = {"Light": "#A7F3D0", "Moderate": "#FDE68A", "Heavy": "#FB923C",
              "Very heavy": "#EF4444", "Extremely heavy": "#7F1D1D"}

def rain_svg(sid):
    s = rain_daily[rain_daily.station_id == sid]
    vals, mx = s.rainfall_mm.values, 250
    bars = "".join(
        f'<rect x="{i*14+2}" y="{100 - v/mx*100:.1f}" width="11" height="{v/mx*100:.1f}" fill="{"#B91C1C" if v >= 115.6 else "#2563EB"}"/>'
        for i, v in enumerate(vals))
    return (f'<svg width="214" height="118"><line x1="0" y1="100" x2="214" y2="100" stroke="#999"/>{bars}'
            f'<text x="2" y="114" font-size="9">1 Jul</text><text x="180" y="114" font-size="9">15 Jul</text></svg>')

rain_layer = folium.FeatureGroup(name="Rainfall stations")
for _, r in rain_gdf.iterrows():
    html = f"""<b>{r.station_name}</b> ({r.station_id})<br>
    Total 1–15 Jul: <b>{r.total_rain_mm:.0f} mm</b><br>
    Peak day: {r.max_daily_mm:.0f} mm — <i>{r.imd_category_peak}</i><br>{rain_svg(r.station_id)}<br>
    <small>Red bars = very heavy rain (≥115.6 mm)</small>"""
    folium.CircleMarker(
        location=[r.lat, r.lon], radius=r.total_rain_mm / 70,
        color="#1F2937", weight=1, fill=True, fill_color=imd_colors[r.imd_category_peak], fill_opacity=0.9,
        tooltip=f"{r.station_name}: {r.total_rain_mm:.0f} mm",
        popup=folium.Popup(html, max_width=240)).add_to(rain_layer)
rain_layer.add_to(m)

# Rainfall intensity heatmap (separate toggle)
plugins.HeatMap(rain_gdf[["lat", "lon", "total_rain_mm"]].values.tolist(),
                name="Rainfall heatmap", radius=45, blur=35, show=False).add_to(m)

In [9]:
status_color = {"Above Danger": "red", "Above Warning": "orange", "Normal": "green"}

def hydro_svg(gid, warn, danger):
    s = gauge_ts[gauge_ts.gauge_id == gid].water_level_m.values
    lo, hi = min(s.min(), warn) - 0.5, max(s.max(), danger) + 0.5
    y = lambda v: 100 - (v - lo) / (hi - lo) * 100
    pts = " ".join(f"{i * 214 / (len(s) - 1):.1f},{y(v):.1f}" for i, v in enumerate(s))
    return (f'<svg width="214" height="112">'
            f'<line x1="0" x2="214" y1="{y(danger):.1f}" y2="{y(danger):.1f}" stroke="#DC2626" stroke-dasharray="4 3"/>'
            f'<line x1="0" x2="214" y1="{y(warn):.1f}" y2="{y(warn):.1f}" stroke="#F59E0B" stroke-dasharray="4 3"/>'
            f'<polyline points="{pts}" fill="none" stroke="#1D4ED8" stroke-width="2"/>'
            f'<text x="2" y="110" font-size="9">1 Jul</text><text x="180" y="110" font-size="9">15 Jul</text></svg>')

gauge_layer = folium.FeatureGroup(name="River gauges")
for _, g in gauge_gdf.iterrows():
    diff = g.water_level_m - g.danger_level_m
    html = f"""<b>{g.site}</b> — {g.river}<br>
    Water level: <b>{g.water_level_m:.2f} m</b> ({g.trend})<br>
    Danger level: {g.danger_level_m:.2f} m &nbsp;|&nbsp; Warning: {g.warning_level_m:.2f} m<br>
    <b style="color:{status_color[g.status]}">{g.status} ({diff:+.2f} m)</b><br>
    {hydro_svg(g.gauge_id, g.warning_level_m, g.danger_level_m)}<br>
    <small>Red dashed = danger, amber = warning · obs. {g.observed_at}</small>"""
    folium.Marker([g.lat, g.lon], tooltip=f"{g.site}: {g.status}",
                  popup=folium.Popup(html, max_width=240),
                  icon=folium.Icon(color=status_color[g.status], icon="tint", prefix="fa")).add_to(gauge_layer)
gauge_layer.add_to(m)

In [10]:
settle_layer = folium.FeatureGroup(name="Settlements (flood exposure)")
for _, s in affected.iterrows():
    col = "#DC2626" if s.flooded else "#16A34A"
    folium.CircleMarker(
        [s.geometry.y, s.geometry.x], radius=5 if s.population < 50000 else 9,
        color="white", weight=1.5, fill=True, fill_color=col, fill_opacity=1,
        tooltip=f"{s['name']} — {s.depth_class}",
        popup=folium.Popup(f"<b>{s['name']}</b> ({s.settlement_type})<br>District: {s.district}<br>"
                           f"Population: {s.population:,}<br>Flood status: <b>{s.depth_class}</b>", max_width=220)
    ).add_to(settle_layer)
settle_layer.add_to(m)

camp_layer = folium.FeatureGroup(name="Relief camps", show=False)
for _, cp in camps.iterrows():
    folium.Marker([cp.lat, cp.lon], tooltip=f"{cp.camp_name} (capacity {cp.capacity})",
                  icon=folium.Icon(color="purple", icon="home", prefix="fa")).add_to(camp_layer)
camp_layer.add_to(m)

In [1]:
import glob
import pandas as pd
import geopandas as gpd
import folium
import branca
from folium import plugins
from branca.element import MacroElement, Template

DATA = "data/"

# ======================= LOAD =======================
study_area   = gpd.read_file(DATA + "study_area.geojson")
settlements  = gpd.read_file(DATA + "settlements.geojson")
rivers       = gpd.read_file(DATA + "rivers.geojson")
normal_water = gpd.read_file(glob.glob(DATA + "normal_water_*.geojson")[0])
flood_extent = gpd.read_file(glob.glob(DATA + "flood_extent_*.geojson")[0])

river_gauges  = pd.read_csv(DATA + "river_gauges.csv")
gauge_levels  = pd.read_csv(DATA + "river_gauge_levels_july2026.csv")
relief_camps  = pd.read_csv(DATA + "relief_camps.csv")
rain_stations = pd.read_csv(DATA + "rainfall_stations.csv")
rain_daily    = pd.read_csv(DATA + "rainfall_daily_july2026.csv")

# ======================= HELPERS =======================
def union(g):
    return g.union_all() if hasattr(g, "union_all") else g.unary_union

def to_wgs84(g):
    return g.to_crs(4326) if g.crs else g.set_crs(4326)

def find_col(df, keys, required=True):
    for k in keys:
        for c in df.columns:
            if k in c.lower():
                return c
    if required:
        raise KeyError(f"No column like {keys}. Columns: {list(df.columns)}")
    return None

def latlon(df):
    return find_col(df, ["latitude", "lat"]), find_col(df, ["longitude", "lon", "lng"])

def shared_key(a, b):
    common = [c for c in a.columns if c in b.columns]
    if not common:
        raise KeyError(f"No shared column between {list(a.columns)} and {list(b.columns)}")
    pref = [c for c in common if any(k in c.lower() for k in ["id", "code", "station", "gauge", "name"])]
    return (pref or common)[0]

def tip(row, cols):
    return "<br>".join(f"<b>{c}</b>: {row[c]}" for c in cols)

# ======================= PREP =======================
study_area, settlements, rivers, normal_water, flood_extent = [
    to_wgs84(g) for g in (study_area, settlements, rivers, normal_water, flood_extent)
]

# --- Areas (in a projected CRS, km²) ---
utm = flood_extent.estimate_utm_crs()
total_flood_km2  = union(flood_extent.to_crs(utm)).area / 1e6
normal_water_km2 = union(normal_water.to_crs(utm)).area / 1e6

# --- Settlements inside the flood extent ---
affected = settlements.copy()
affected["geometry"] = settlements.geometry.representative_point()
affected["flooded"] = affected.within(union(flood_extent))
pop_col = find_col(affected, ["pop"], required=False)
affected["population"] = (pd.to_numeric(affected[pop_col], errors="coerce").fillna(0)
                          if pop_col else 0)
name_col = find_col(affected, ["name"], required=False) or affected.columns[0]
exposed_pop = affected.loc[affected.flooded, "population"].sum()

# --- Rainfall: peak day per station ---
rs_lat, rs_lon = latlon(rain_stations)
rkey = shared_key(rain_stations, rain_daily)
rain_col = find_col(rain_daily, ["rain", "precip", "mm"])
rain_daily[rain_col] = pd.to_numeric(rain_daily[rain_col], errors="coerce")
date_col = find_col(rain_daily, ["date", "day"], required=False)
peak = rain_daily.loc[rain_daily.groupby(rkey)[rain_col].idxmax()]
keep = [rkey, rain_col] + ([date_col] if date_col and date_col != rain_col else [])
peak = peak[keep].rename(columns={rain_col: "peak_mm", **({date_col: "peak_date"} if date_col else {})})
rain_pts = rain_stations.merge(peak, on=rkey, how="left")

def rain_class(mm):
    if pd.isna(mm):  return "#9CA3AF", "No data"
    if mm > 204.4:   return "#7F1D1D", "Extreme"
    if mm > 115.5:   return "#EF4444", "Very heavy"
    if mm >= 64.5:   return "#FB923C", "Heavy"
    return "#FDE68A", "Below heavy"

# --- River gauges: peak level vs danger / warning ---
g_lat, g_lon = latlon(river_gauges)
gkey = shared_key(river_gauges, gauge_levels)
lvl_col = find_col(gauge_levels, ["level", "stage", "water"])
gauge_levels[lvl_col] = pd.to_numeric(gauge_levels[lvl_col], errors="coerce")
peak_g = gauge_levels.groupby(gkey)[lvl_col].max().rename("peak_level").reset_index()
gauges = river_gauges.merge(peak_g, on=gkey, how="left")
danger_col = find_col(gauges, ["danger"], required=False)
warn_col   = find_col(gauges, ["warn"], required=False)

def gauge_status(r):
    lv = r["peak_level"]
    if pd.isna(lv): return "#9CA3AF", "No data"
    if danger_col and lv >= pd.to_numeric(r[danger_col], errors="coerce"): return "#DC2626", "Danger"
    if warn_col and lv >= pd.to_numeric(r[warn_col], errors="coerce"):     return "#F59E0B", "Warning"
    return "#16A34A", "Normal"

# ======================= MAP =======================
minx, miny, maxx, maxy = study_area.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2],
               zoom_start=10, tiles="CartoDB positron")

folium.GeoJson(study_area, name="Study area",
    style_function=lambda f: {"color": "black", "weight": 2, "fillOpacity": 0}).add_to(m)

folium.GeoJson(normal_water, name="Normal water (20 May 2026)",
    style_function=lambda f: {"color": "#06B6D4", "fillColor": "#06B6D4",
                              "fillOpacity": 0.6, "weight": 1}).add_to(m)

# Flood extent, coloured by depth if a depth column exists
depth_col = find_col(flood_extent, ["depth"], required=False)

def flood_style(f):
    color = "#3B82F6"
    if depth_col:
        try:
            v = float(f["properties"].get(depth_col))
            color = "#93C5FD" if v < 0.5 else "#3B82F6" if v <= 1.5 else "#1E3A8A"
        except (TypeError, ValueError):
            pass
    return {"color": color, "fillColor": color, "fillOpacity": 0.6, "weight": 0.5}

folium.GeoJson(flood_extent, name="Flood extent (10 Jul 2026)",
               style_function=flood_style).add_to(m)

folium.GeoJson(rivers, name="Rivers",
    style_function=lambda f: {"color": "#0B5394", "weight": 2}).add_to(m)

# Settlements: red = flooded, green = safe
fg = folium.FeatureGroup(name="Settlements")
for _, r in affected.iterrows():
    folium.CircleMarker(
        [r.geometry.y, r.geometry.x], radius=4,
        color="#DC2626" if r.flooded else "#16A34A",
        fill=True, fill_opacity=0.9,
        tooltip=folium.Tooltip(f"<b>{r[name_col]}</b><br>Flooded: {r.flooded}"
                               f"<br>Population: {r.population:,.0f}")).add_to(fg)
fg.add_to(m)

# Rainfall stations (peak day)
fg = folium.FeatureGroup(name="Rainfall stations (peak day)")
for _, r in rain_pts.dropna(subset=[rs_lat, rs_lon]).iterrows():
    color, cls = rain_class(r["peak_mm"])
    folium.CircleMarker(
        [r[rs_lat], r[rs_lon]], radius=7, color="#333", weight=1,
        fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=folium.Tooltip(f"{tip(r, rain_stations.columns[:4])}"
                               f"<br><b>Peak</b>: {r['peak_mm']} mm ({cls})")).add_to(fg)
fg.add_to(m)

# River gauges
fg = folium.FeatureGroup(name="River gauges")
for _, r in gauges.dropna(subset=[g_lat, g_lon]).iterrows():
    color, status = gauge_status(r)
    folium.CircleMarker(
        [r[g_lat], r[g_lon]], radius=8, color="#333", weight=1,
        fill=True, fill_color=color, fill_opacity=0.95,
        tooltip=folium.Tooltip(f"{tip(r, river_gauges.columns[:4])}"
                               f"<br><b>Peak level</b>: {r['peak_level']}"
                               f"<br><b>Status</b>: {status}")).add_to(fg)
fg.add_to(m)

# Relief camps
c_lat, c_lon = latlon(relief_camps)
fg = folium.FeatureGroup(name="Relief camps")
for _, r in relief_camps.dropna(subset=[c_lat, c_lon]).iterrows():
    folium.Marker([r[c_lat], r[c_lon]],
                  tooltip=folium.Tooltip(tip(r, relief_camps.columns[:6])),
                  icon=folium.Icon(color="green", icon="home")).add_to(fg)
fg.add_to(m)

m.fit_bounds([[miny, minx], [maxy, maxx]])

# ======================= CONTROLS =======================
plugins.Fullscreen().add_to(m)
plugins.MiniMap(toggle_display=True, position="bottomleft").add_to(m)
plugins.MeasureControl(primary_length_unit="kilometers",
                       primary_area_unit="sqkilometers").add_to(m)
plugins.MousePosition(position="bottomright", prefix="Lat/Lon:").add_to(m)

# ======================= TITLE =======================
title_html = f"""
<div style="position:fixed; top:10px; left:55px; z-index:9999; background:white; padding:8px 14px;
     border-left:5px solid #F59E0B; border-radius:4px; box-shadow:0 1px 6px rgba(0,0,0,.3); font-family:Arial">
  <div style="font-size:16px; font-weight:bold; color:#0F172A">Central Assam Flood Monitoring – 10 July 2026</div>
  <div style="font-size:12px; color:#334155">Flooded: {total_flood_km2:,.0f} km² · Settlements affected:
     {affected.flooded.sum()}/{len(affected)} · Population exposed: {exposed_pop:,.0f}</div>
</div>"""
m.get_root().html.add_child(folium.Element(title_html))

# ======================= LEGEND =======================
legend_html = """
{% macro html(this, kwargs) %}
<div style="position:fixed; bottom:30px; right:10px; z-index:9999; background:white; padding:10px 12px;
     border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.3); font:12px Arial; line-height:18px">
<b>Flood depth</b><br>
<i style="background:#93C5FD;width:14px;height:10px;display:inline-block"></i> Shallow &lt;0.5 m<br>
<i style="background:#3B82F6;width:14px;height:10px;display:inline-block"></i> Moderate 0.5–1.5 m<br>
<i style="background:#1E3A8A;width:14px;height:10px;display:inline-block"></i> Deep &gt;1.5 m<br>
<i style="background:#06B6D4;width:14px;height:10px;display:inline-block"></i> Normal water<br>
<b>Rainfall (peak day, IMD)</b><br>
<i style="background:#FDE68A;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Below heavy<br>
<i style="background:#FB923C;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Heavy<br>
<i style="background:#EF4444;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Very heavy<br>
<i style="background:#7F1D1D;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Extreme<br>
<b>Gauges</b> &nbsp;<span style="color:#DC2626">●</span> Danger <span style="color:#F59E0B">●</span> Warning <span style="color:#16A34A">●</span> Normal<br>
<b>Settlements</b> &nbsp;<span style="color:#DC2626">●</span> Flooded <span style="color:#16A34A">●</span> Safe
</div>
{% endmacro %}"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl(collapsed=False).add_to(m)

# ======================= SAVE & SHOW =======================
m.save("flood_map.html")
print(f"Flood area: {total_flood_km2:,.1f} km² | Normal water: {normal_water_km2:,.1f} km²")
print(f"Settlements flooded: {affected.flooded.sum()} of {len(affected)} | Population exposed: {exposed_pop:,.0f}")
print("Detected -> population:", pop_col, "| depth:", depth_col, "| rainfall:", rain_col,
      "| gauge level:", lvl_col, "| danger:", danger_col, "| warning:", warn_col)
m

Flood area: 4,531.9 km² | Normal water: 710.3 km²
Settlements flooded: 7 of 20 | Population exposed: 204,000
Detected -> population: population | depth: depth_class | rainfall: rainfall_mm | gauge level: water_level_m | danger: danger_level_m | warning: warning_level_m


C:\Users\aafir\AppData\Local\Programs\Python\Python313\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [2]:
m.save("Assam_Flood_Monitoring_Map.html")
print("Saved: Assam_Flood_Monitoring_Map.html")

Saved: Assam_Flood_Monitoring_Map.html


In [3]:
plugins.Fullscreen().add_to(m)
plugins.MiniMap(toggle_display=True, position="bottomleft").add_to(m)
plugins.MeasureControl(primary_length_unit="kilometers", primary_area_unit="sqkilometers").add_to(m)
plugins.MousePosition(position="bottomright", prefix="Lat/Lon:").add_to(m)

exposed_pop = affected.loc[affected.flooded, "population"].sum()
title_html = f"""
<div style="position:fixed; top:10px; left:55px; z-index:9999; background:white; padding:8px 14px;
     border-left:5px solid #F59E0B; border-radius:4px; box-shadow:0 1px 6px rgba(0,0,0,.3); font-family:Arial">
  <div style="font-size:16px; font-weight:bold; color:#0F172A">Central Assam Flood Monitoring — 10 July 2026</div>
  <div style="font-size:12px; color:#334155">Flooded: {total_flood_km2:,.0f} km² · Settlements affected:
       {affected.flooded.sum()}/{len(affected)} · Population exposed: {exposed_pop:,}</div>
</div>"""
m.get_root().html.add_child(folium.Element(title_html))

legend_html = """
{% macro html(this, kwargs) %}
<div style="position:fixed; bottom:30px; right:10px; z-index:9999; background:white; padding:10px 12px;
     border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.3); font:12px Arial; line-height:18px">
<b>Flood depth</b><br>
<i style="background:#93C5FD;width:14px;height:10px;display:inline-block"></i> Shallow &lt;0.5 m<br>
<i style="background:#3B82F6;width:14px;height:10px;display:inline-block"></i> Moderate 0.5–1.5 m<br>
<i style="background:#1E3A8A;width:14px;height:10px;display:inline-block"></i> Deep &gt;1.5 m<br>
<i style="background:#06B6D4;width:14px;height:10px;display:inline-block"></i> Normal water<br>
<b>Rainfall (peak day, IMD)</b><br>
<i style="background:#FB923C;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Heavy
<i style="background:#EF4444;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Very heavy
<i style="background:#7F1D1D;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Extreme<br>
<b>Gauges</b> &nbsp;<span style="color:#DC2626">●</span> Danger <span style="color:#F59E0B">●</span> Warning
<span style="color:#16A34A">●</span> Normal<br>
<b>Settlements</b> &nbsp;<span style="color:#DC2626">●</span> Flooded <span style="color:#16A34A">●</span> Safe
</div>
{% endmacro %}"""
legend = branca.element.MacroElement()
legend._template = branca.element.Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl(collapsed=False).add_to(m)
m


In [4]:
import glob
import pandas as pd
import geopandas as gpd
import folium
import branca
from folium import plugins
from branca.element import MacroElement, Template

DATA = "data/"

# ======================= LOAD =======================
study_area   = gpd.read_file(DATA + "study_area.geojson")
settlements  = gpd.read_file(DATA + "settlements.geojson")
rivers       = gpd.read_file(DATA + "rivers.geojson")
normal_water = gpd.read_file(glob.glob(DATA + "normal_water_*.geojson")[0])
flood_extent = gpd.read_file(glob.glob(DATA + "flood_extent_*.geojson")[0])

river_gauges  = pd.read_csv(DATA + "river_gauges.csv")
gauge_levels  = pd.read_csv(DATA + "river_gauge_levels_july2026.csv")
relief_camps  = pd.read_csv(DATA + "relief_camps.csv")
rain_stations = pd.read_csv(DATA + "rainfall_stations.csv")
rain_daily    = pd.read_csv(DATA + "rainfall_daily_july2026.csv")

# ======================= HELPERS =======================
def union(g):
    return g.union_all() if hasattr(g, "union_all") else g.unary_union

def to_wgs84(g):
    return g.to_crs(4326) if g.crs else g.set_crs(4326)

def find_col(df, keys, required=True):
    for k in keys:
        for c in df.columns:
            if k in c.lower():
                return c
    if required:
        raise KeyError(f"No column like {keys}. Columns: {list(df.columns)}")
    return None

def latlon(df):
    return find_col(df, ["latitude", "lat"]), find_col(df, ["longitude", "lon", "lng"])

def shared_key(a, b):
    common = [c for c in a.columns if c in b.columns]
    if not common:
        raise KeyError(f"No shared column between {list(a.columns)} and {list(b.columns)}")
    pref = [c for c in common if any(k in c.lower() for k in ["id", "code", "station", "gauge", "name"])]
    return (pref or common)[0]

def tip(row, cols):
    return "<br>".join(f"<b>{c}</b>: {row[c]}" for c in cols)

# ======================= PREP =======================
study_area, settlements, rivers, normal_water, flood_extent = [
    to_wgs84(g) for g in (study_area, settlements, rivers, normal_water, flood_extent)
]

# --- Areas (projected CRS, km²) ---
utm = flood_extent.estimate_utm_crs()
total_flood_km2  = union(flood_extent.to_crs(utm)).area / 1e6
normal_water_km2 = union(normal_water.to_crs(utm)).area / 1e6

# --- Settlements inside the flood extent ---
affected = settlements.copy()
affected["geometry"] = settlements.geometry.representative_point()
affected["flooded"] = affected.within(union(flood_extent))
pop_col = find_col(affected, ["pop"], required=False)
affected["population"] = (pd.to_numeric(affected[pop_col], errors="coerce").fillna(0)
                          if pop_col else 0)
name_col = find_col(affected, ["name"], required=False) or affected.columns[0]
exposed_pop = affected.loc[affected.flooded, "population"].sum()

# --- Rainfall: peak day per station ---
rs_lat, rs_lon = latlon(rain_stations)
rkey = shared_key(rain_stations, rain_daily)
rain_col = find_col(rain_daily, ["rain", "precip", "mm"])
rain_daily[rain_col] = pd.to_numeric(rain_daily[rain_col], errors="coerce")
date_col = find_col(rain_daily, ["date", "day"], required=False)
peak = rain_daily.loc[rain_daily.groupby(rkey)[rain_col].idxmax()]
keep = [rkey, rain_col] + ([date_col] if date_col and date_col != rain_col else [])
peak = peak[keep].rename(columns={rain_col: "peak_mm", **({date_col: "peak_date"} if date_col else {})})
rain_pts = rain_stations.merge(peak, on=rkey, how="left")

def rain_class(mm):
    if pd.isna(mm):  return "#9CA3AF", "No data"
    if mm > 204.4:   return "#7F1D1D", "Extreme"
    if mm > 115.5:   return "#EF4444", "Very heavy"
    if mm >= 64.5:   return "#FB923C", "Heavy"
    return "#FDE68A", "Below heavy"

# --- River gauges: peak level vs danger / warning ---
g_lat, g_lon = latlon(river_gauges)
gkey = shared_key(river_gauges, gauge_levels)
lvl_col = find_col(gauge_levels, ["level", "stage", "water"])
gauge_levels[lvl_col] = pd.to_numeric(gauge_levels[lvl_col], errors="coerce")
peak_g = gauge_levels.groupby(gkey)[lvl_col].max().rename("peak_level").reset_index()
gauges = river_gauges.merge(peak_g, on=gkey, how="left")
danger_col = find_col(gauges, ["danger"], required=False)
warn_col   = find_col(gauges, ["warn"], required=False)

def gauge_status(r):
    lv = r["peak_level"]
    if pd.isna(lv): return "#9CA3AF", "No data"
    if danger_col and lv >= pd.to_numeric(r[danger_col], errors="coerce"): return "#DC2626", "Danger"
    if warn_col and lv >= pd.to_numeric(r[warn_col], errors="coerce"):     return "#F59E0B", "Warning"
    return "#16A34A", "Normal"

# ======================= MAP + REAL BASEMAPS =======================
minx, miny, maxx, maxy = study_area.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2],
               zoom_start=10, tiles=None)

folium.TileLayer("OpenStreetMap", name="Street map (OSM)", show=True).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite (Esri)", show=False).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Topo", name="Topographic (Esri)", show=False).add_to(m)

folium.TileLayer("CartoDB positron", name="Light grey", show=False).add_to(m)

# ======================= DATA LAYERS =======================
folium.GeoJson(study_area, name="Study area",
    style_function=lambda f: {"color": "black", "weight": 2, "fillOpacity": 0}).add_to(m)

folium.GeoJson(normal_water, name="Normal water (20 May 2026)",
    style_function=lambda f: {"color": "#06B6D4", "fillColor": "#06B6D4",
                              "fillOpacity": 0.55, "weight": 1}).add_to(m)

depth_col = find_col(flood_extent, ["depth"], required=False)

def flood_style(f):
    color = "#3B82F6"
    if depth_col:
        try:
            v = float(f["properties"].get(depth_col))
            color = "#93C5FD" if v < 0.5 else "#3B82F6" if v <= 1.5 else "#1E3A8A"
        except (TypeError, ValueError):
            pass
    return {"color": color, "fillColor": color, "fillOpacity": 0.5, "weight": 0.5}

folium.GeoJson(flood_extent, name="Flood extent (10 Jul 2026)",
               style_function=flood_style).add_to(m)

folium.GeoJson(rivers, name="Rivers",
    style_function=lambda f: {"color": "#0B5394", "weight": 2}).add_to(m)

# Settlements: red = flooded, green = safe
fg = folium.FeatureGroup(name="Settlements")
for _, r in affected.iterrows():
    folium.CircleMarker(
        [r.geometry.y, r.geometry.x], radius=4,
        color="#DC2626" if r.flooded else "#16A34A",
        fill=True, fill_opacity=0.9,
        tooltip=folium.Tooltip(f"<b>{r[name_col]}</b><br>Flooded: {r.flooded}"
                               f"<br>Population: {r.population:,.0f}")).add_to(fg)
fg.add_to(m)

# Rainfall stations (peak day)
fg = folium.FeatureGroup(name="Rainfall stations (peak day)")
for _, r in rain_pts.dropna(subset=[rs_lat, rs_lon]).iterrows():
    color, cls = rain_class(r["peak_mm"])
    folium.CircleMarker(
        [r[rs_lat], r[rs_lon]], radius=7, color="#333", weight=1,
        fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=folium.Tooltip(f"{tip(r, rain_stations.columns[:4])}"
                               f"<br><b>Peak</b>: {r['peak_mm']} mm ({cls})")).add_to(fg)
fg.add_to(m)

# River gauges
fg = folium.FeatureGroup(name="River gauges")
for _, r in gauges.dropna(subset=[g_lat, g_lon]).iterrows():
    color, status = gauge_status(r)
    folium.CircleMarker(
        [r[g_lat], r[g_lon]], radius=8, color="#333", weight=1,
        fill=True, fill_color=color, fill_opacity=0.95,
        tooltip=folium.Tooltip(f"{tip(r, river_gauges.columns[:4])}"
                               f"<br><b>Peak level</b>: {r['peak_level']}"
                               f"<br><b>Status</b>: {status}")).add_to(fg)
fg.add_to(m)

# Relief camps
c_lat, c_lon = latlon(relief_camps)
fg = folium.FeatureGroup(name="Relief camps")
for _, r in relief_camps.dropna(subset=[c_lat, c_lon]).iterrows():
    folium.Marker([r[c_lat], r[c_lon]],
                  tooltip=folium.Tooltip(tip(r, relief_camps.columns[:6])),
                  icon=folium.Icon(color="green", icon="home")).add_to(fg)
fg.add_to(m)

m.fit_bounds([[miny, minx], [maxy, maxx]])

# ======================= CONTROLS =======================
plugins.Fullscreen().add_to(m)
plugins.MiniMap(toggle_display=True, position="bottomleft").add_to(m)
plugins.MeasureControl(primary_length_unit="kilometers",
                       primary_area_unit="sqkilometers").add_to(m)
plugins.MousePosition(position="bottomright", prefix="Lat/Lon:").add_to(m)

# ======================= TITLE =======================
title_html = f"""
<div style="position:fixed; top:10px; left:55px; z-index:9999; background:white; padding:8px 14px;
     border-left:5px solid #F59E0B; border-radius:4px; box-shadow:0 1px 6px rgba(0,0,0,.3); font-family:Arial">
  <div style="font-size:16px; font-weight:bold; color:#0F172A">Central Assam Flood Monitoring – 10 July 2026</div>
  <div style="font-size:12px; color:#334155">Flooded: {total_flood_km2:,.0f} km² · Settlements affected:
     {affected.flooded.sum()}/{len(affected)} · Population exposed: {exposed_pop:,.0f}</div>
</div>"""
m.get_root().html.add_child(folium.Element(title_html))

# ======================= LEGEND =======================
legend_html = """
{% macro html(this, kwargs) %}
<div style="position:fixed; bottom:30px; right:10px; z-index:9999; background:white; padding:10px 12px;
     border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.3); font:12px Arial; line-height:18px">
<b>Flood depth</b><br>
<i style="background:#93C5FD;width:14px;height:10px;display:inline-block"></i> Shallow &lt;0.5 m<br>
<i style="background:#3B82F6;width:14px;height:10px;display:inline-block"></i> Moderate 0.5–1.5 m<br>
<i style="background:#1E3A8A;width:14px;height:10px;display:inline-block"></i> Deep &gt;1.5 m<br>
<i style="background:#06B6D4;width:14px;height:10px;display:inline-block"></i> Normal water<br>
<b>Rainfall (peak day, IMD)</b><br>
<i style="background:#FDE68A;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Below heavy<br>
<i style="background:#FB923C;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Heavy<br>
<i style="background:#EF4444;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Very heavy<br>
<i style="background:#7F1D1D;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Extreme<br>
<b>Gauges</b> &nbsp;<span style="color:#DC2626">●</span> Danger <span style="color:#F59E0B">●</span> Warning <span style="color:#16A34A">●</span> Normal<br>
<b>Settlements</b> &nbsp;<span style="color:#DC2626">●</span> Flooded <span style="color:#16A34A">●</span> Safe
</div>
{% endmacro %}"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl(collapsed=False).add_to(m)

# ======================= SAVE & SHOW =======================
m.save("flood_map.html")
print(f"Flood area: {total_flood_km2:,.1f} km² | Normal water: {normal_water_km2:,.1f} km²")
print(f"Settlements flooded: {affected.flooded.sum()} of {len(affected)} | Population exposed: {exposed_pop:,.0f}")
print("Detected -> population:", pop_col, "| depth:", depth_col, "| rainfall:", rain_col,
      "| gauge level:", lvl_col, "| danger:", danger_col, "| warning:", warn_col)
m

C:\Users\aafir\AppData\Local\Programs\Python\Python313\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


Flood area: 4,531.9 km² | Normal water: 710.3 km²
Settlements flooded: 7 of 20 | Population exposed: 204,000
Detected -> population: population | depth: depth_class | rainfall: rainfall_mm | gauge level: water_level_m | danger: danger_level_m | warning: warning_level_m


In [5]:
import glob
import pandas as pd
import geopandas as gpd
import folium
import branca
from folium import plugins
from branca.element import MacroElement, Template

DATA = "data/"

# ======================= LOAD =======================
study_area   = gpd.read_file(DATA + "study_area.geojson")
settlements  = gpd.read_file(DATA + "settlements.geojson")
rivers       = gpd.read_file(DATA + "rivers.geojson")
normal_water = gpd.read_file(glob.glob(DATA + "normal_water_*.geojson")[0])
flood_extent = gpd.read_file(glob.glob(DATA + "flood_extent_*.geojson")[0])

river_gauges  = pd.read_csv(DATA + "river_gauges.csv")
gauge_levels  = pd.read_csv(DATA + "river_gauge_levels_july2026.csv")
relief_camps  = pd.read_csv(DATA + "relief_camps.csv")
rain_stations = pd.read_csv(DATA + "rainfall_stations.csv")
rain_daily    = pd.read_csv(DATA + "rainfall_daily_july2026.csv")

# ======================= HELPERS =======================
def union(g):
    return g.union_all() if hasattr(g, "union_all") else g.unary_union

def to_wgs84(g):
    return g.to_crs(4326) if g.crs else g.set_crs(4326)

def find_col(df, keys, required=True):
    for k in keys:
        for c in df.columns:
            if k in c.lower():
                return c
    if required:
        raise KeyError(f"No column like {keys}. Columns: {list(df.columns)}")
    return None

def latlon(df):
    return find_col(df, ["latitude", "lat"]), find_col(df, ["longitude", "lon", "lng"])

def shared_key(a, b):
    common = [c for c in a.columns if c in b.columns]
    if not common:
        raise KeyError(f"No shared column between {list(a.columns)} and {list(b.columns)}")
    pref = [c for c in common if any(k in c.lower() for k in ["id", "code", "station", "gauge", "name"])]
    return (pref or common)[0]

def tip(row, cols):
    return "<br>".join(f"<b>{c}</b>: {row[c]}" for c in cols)

# ======================= PREP =======================
study_area, settlements, rivers, normal_water, flood_extent = [
    to_wgs84(g) for g in (study_area, settlements, rivers, normal_water, flood_extent)
]

# --- Areas (projected CRS, km²) ---
utm = flood_extent.estimate_utm_crs()
total_flood_km2  = union(flood_extent.to_crs(utm)).area / 1e6
normal_water_km2 = union(normal_water.to_crs(utm)).area / 1e6

# --- Settlements inside the flood extent ---
affected = settlements.copy()
affected["geometry"] = settlements.geometry.representative_point()
affected["flooded"] = affected.within(union(flood_extent))
pop_col = find_col(affected, ["pop"], required=False)
affected["population"] = (pd.to_numeric(affected[pop_col], errors="coerce").fillna(0)
                          if pop_col else 0)
name_col = find_col(affected, ["name"], required=False) or affected.columns[0]
exposed_pop = affected.loc[affected.flooded, "population"].sum()

# --- Rainfall: peak day per station ---
rs_lat, rs_lon = latlon(rain_stations)
rkey = shared_key(rain_stations, rain_daily)
rain_col = find_col(rain_daily, ["rain", "precip", "mm"])
rain_daily[rain_col] = pd.to_numeric(rain_daily[rain_col], errors="coerce")
date_col = find_col(rain_daily, ["date", "day"], required=False)
peak = rain_daily.loc[rain_daily.groupby(rkey)[rain_col].idxmax()]
keep = [rkey, rain_col] + ([date_col] if date_col and date_col != rain_col else [])
peak = peak[keep].rename(columns={rain_col: "peak_mm", **({date_col: "peak_date"} if date_col else {})})
rain_pts = rain_stations.merge(peak, on=rkey, how="left")

def rain_class(mm):
    if pd.isna(mm):  return "#9CA3AF", "No data"
    if mm > 204.4:   return "#7F1D1D", "Extreme"
    if mm > 115.5:   return "#EF4444", "Very heavy"
    if mm >= 64.5:   return "#FB923C", "Heavy"
    return "#FDE68A", "Below heavy"

# --- River gauges: peak level vs danger / warning ---
g_lat, g_lon = latlon(river_gauges)
gkey = shared_key(river_gauges, gauge_levels)
lvl_col = find_col(gauge_levels, ["level", "stage", "water"])
gauge_levels[lvl_col] = pd.to_numeric(gauge_levels[lvl_col], errors="coerce")
peak_g = gauge_levels.groupby(gkey)[lvl_col].max().rename("peak_level").reset_index()
gauges = river_gauges.merge(peak_g, on=gkey, how="left")
danger_col = find_col(gauges, ["danger"], required=False)
warn_col   = find_col(gauges, ["warn"], required=False)

def gauge_status(r):
    lv = r["peak_level"]
    if pd.isna(lv): return "#9CA3AF", "No data"
    if danger_col and lv >= pd.to_numeric(r[danger_col], errors="coerce"): return "#DC2626", "Danger"
    if warn_col and lv >= pd.to_numeric(r[warn_col], errors="coerce"):     return "#F59E0B", "Warning"
    return "#16A34A", "Normal"

# ======================= MAP + BASEMAPS =======================
minx, miny, maxx, maxy = study_area.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2],
               zoom_start=10, tiles=None, control_scale=True)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap", show=False).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin", name="Light grey (Esri)", show=False).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite (Esri)", show=True).add_to(m)

# ======================= DATA LAYERS =======================
folium.GeoJson(study_area, name="Study area",
    style_function=lambda f: {"color": "#FFFFFF", "weight": 2.5, "fillOpacity": 0}).add_to(m)

folium.GeoJson(normal_water, name="Normal water (20 May 2026)",
    style_function=lambda f: {"color": "#06B6D4", "fillColor": "#06B6D4",
                              "fillOpacity": 0.6, "weight": 1}).add_to(m)

depth_col = find_col(flood_extent, ["depth"], required=False)

def flood_style(f):
    color = "#3B82F6"
    if depth_col:
        try:
            v = float(f["properties"].get(depth_col))
            color = "#93C5FD" if v < 0.5 else "#3B82F6" if v <= 1.5 else "#1E3A8A"
        except (TypeError, ValueError):
            pass
    return {"color": color, "fillColor": color, "fillOpacity": 0.55, "weight": 0.5}

folium.GeoJson(flood_extent, name="Flood extent (10 Jul 2026)",
               style_function=flood_style).add_to(m)

folium.GeoJson(rivers, name="River network",
    style_function=lambda f: {"color": "#38BDF8", "weight": 2}).add_to(m)

# Rainfall stations (peak day)
fg = folium.FeatureGroup(name="Rainfall stations")
for _, r in rain_pts.dropna(subset=[rs_lat, rs_lon]).iterrows():
    color, cls = rain_class(r["peak_mm"])
    folium.CircleMarker(
        [r[rs_lat], r[rs_lon]], radius=7, color="#333", weight=1,
        fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=folium.Tooltip(f"{tip(r, rain_stations.columns[:4])}"
                               f"<br><b>Peak</b>: {r['peak_mm']} mm ({cls})")).add_to(fg)
fg.add_to(m)

# Rainfall heatmap (weighted by peak-day rainfall)
heat_df = rain_pts.dropna(subset=[rs_lat, rs_lon, "peak_mm"])
if len(heat_df):
    wmax = heat_df["peak_mm"].max() or 1
    heat_data = [[r[rs_lat], r[rs_lon], r["peak_mm"] / wmax] for _, r in heat_df.iterrows()]
    plugins.HeatMap(heat_data, name="Rainfall heatmap", radius=45, blur=35, min_opacity=0.3,
                    gradient={0.2: "#FDE68A", 0.5: "#FB923C", 0.8: "#EF4444", 1.0: "#7F1D1D"}
                    ).add_to(m)

# River gauges
fg = folium.FeatureGroup(name="River gauges")
for _, r in gauges.dropna(subset=[g_lat, g_lon]).iterrows():
    color, status = gauge_status(r)
    folium.CircleMarker(
        [r[g_lat], r[g_lon]], radius=8, color="#333", weight=1,
        fill=True, fill_color=color, fill_opacity=0.95,
        tooltip=folium.Tooltip(f"{tip(r, river_gauges.columns[:4])}"
                               f"<br><b>Peak level</b>: {r['peak_level']}"
                               f"<br><b>Status</b>: {status}")).add_to(fg)
fg.add_to(m)

# Settlements: red = flooded, green = safe
fg = folium.FeatureGroup(name="Settlements (flood exposure)")
for _, r in affected.iterrows():
    folium.CircleMarker(
        [r.geometry.y, r.geometry.x], radius=4,
        color="#DC2626" if r.flooded else "#16A34A",
        fill=True, fill_opacity=0.9,
        tooltip=folium.Tooltip(f"<b>{r[name_col]}</b><br>Flooded: {r.flooded}"
                               f"<br>Population: {r.population:,.0f}")).add_to(fg)
fg.add_to(m)

# Relief camps
c_lat, c_lon = latlon(relief_camps)
fg = folium.FeatureGroup(name="Relief camps")
for _, r in relief_camps.dropna(subset=[c_lat, c_lon]).iterrows():
    folium.Marker([r[c_lat], r[c_lon]],
                  tooltip=folium.Tooltip(tip(r, relief_camps.columns[:6])),
                  icon=folium.Icon(color="green", icon="home")).add_to(fg)
fg.add_to(m)

m.fit_bounds([[miny, minx], [maxy, maxx]])

# ======================= CONTROLS =======================
plugins.Fullscreen().add_to(m)
plugins.MiniMap(toggle_display=True, position="bottomleft").add_to(m)
plugins.MeasureControl(primary_length_unit="kilometers",
                       primary_area_unit="sqkilometers").add_to(m)
plugins.MousePosition(position="bottomright", prefix="Lat/Lon:").add_to(m)

# ======================= TITLE =======================
title_html = f"""
<div style="position:fixed; top:10px; left:55px; z-index:9999; background:white; padding:8px 14px;
     border-left:5px solid #F59E0B; border-radius:4px; box-shadow:0 1px 6px rgba(0,0,0,.3); font-family:Arial">
  <div style="font-size:16px; font-weight:bold; color:#0F172A">Central Assam Flood Monitoring – 10 July 2026</div>
  <div style="font-size:12px; color:#334155">Flooded: {total_flood_km2:,.0f} km² · Settlements affected:
     {affected.flooded.sum()}/{len(affected)} · Population exposed: {exposed_pop:,.0f}</div>
</div>"""
m.get_root().html.add_child(folium.Element(title_html))

# ======================= LEGEND =======================
legend_html = """
{% macro html(this, kwargs) %}
<div style="position:fixed; bottom:30px; right:10px; z-index:9999; background:white; padding:10px 12px;
     border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.3); font:12px Arial; line-height:18px">
<b>Flood depth</b><br>
<i style="background:#93C5FD;width:14px;height:10px;display:inline-block"></i> Shallow &lt;0.5 m<br>
<i style="background:#3B82F6;width:14px;height:10px;display:inline-block"></i> Moderate 0.5–1.5 m<br>
<i style="background:#1E3A8A;width:14px;height:10px;display:inline-block"></i> Deep &gt;1.5 m<br>
<i style="background:#06B6D4;width:14px;height:10px;display:inline-block"></i> Normal water<br>
<b>Rainfall (peak day, IMD)</b><br>
<i style="background:#FDE68A;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Below heavy<br>
<i style="background:#FB923C;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Heavy<br>
<i style="background:#EF4444;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Very heavy<br>
<i style="background:#7F1D1D;border-radius:50%;width:10px;height:10px;display:inline-block"></i> Extreme<br>
<b>Gauges</b> &nbsp;<span style="color:#DC2626">●</span> Danger <span style="color:#F59E0B">●</span> Warning <span style="color:#16A34A">●</span> Normal<br>
<b>Settlements</b> &nbsp;<span style="color:#DC2626">●</span> Flooded <span style="color:#16A34A">●</span> Safe
</div>
{% endmacro %}"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl(collapsed=False).add_to(m)

# ======================= SAVE & SHOW =======================
m.save("flood_map.html")
print(f"Flood area: {total_flood_km2:,.1f} km² | Normal water: {normal_water_km2:,.1f} km²")
print(f"Settlements flooded: {affected.flooded.sum()} of {len(affected)} | Population exposed: {exposed_pop:,.0f}")
print("Detected -> population:", pop_col, "| depth:", depth_col, "| rainfall:", rain_col,
      "| gauge level:", lvl_col, "| danger:", danger_col, "| warning:", warn_col)
m

Flood area: 4,531.9 km² | Normal water: 710.3 km²
Settlements flooded: 7 of 20 | Population exposed: 204,000
Detected -> population: population | depth: depth_class | rainfall: rainfall_mm | gauge level: water_level_m | danger: danger_level_m | warning: warning_level_m
